In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


In [2]:
data = pd.read_csv("Accruals Workable.csv")
data.head()

,gvkey,Date,Company Name,Fiscal Year,Total Current Assets,Total Assets,Total Long-Term Debt,Accumulated Depreciation,Total Current Liabilities,Gross Property Plant and Equipment,Total Receivables,Total Trade Receivables,COGS,Net Income,Total Revenue,Advertising Expense,R&D Expense,SG&A Expense,Net Cash Flow from Operations,Year
0,6266,1987-12-31,JOHNSON & JOHNSON,12,0.499847,6546.0,0.111977,0.212649,0.269325,0.556370,0.145585,0.145585,0.404064,0.127253,1.223954,0.067980,0.094256,0.587382,0.167125,1987
1,6266,1988-12-31,JOHNSON & JOHNSON,12,0.492063,7119.0,0.163787,0.207332,0.262396,0.557522,0.159433,0.159433,0.415086,0.136817,1.264223,0.065880,0.094676,0.604579,0.124737,1988
2,7257,1988-12-31,MERCK & CO INC,12,0.553129,6127.5,0.023305,0.248029,0.311546,0.585965,0.166920,0.166920,0.218213,0.196948,0.969319,0.039412,0.109147,0.415602,0.226324,1988
3,6266,1989-12-31,JOHNSON & JOHNSON,12,0.476828,7919.0,0.147746,0.211011,0.243339,0.570400,0.166688,0.166688,0.394242,0.136633,1.232100,0.059477,0.090794,0.582902,0.157848,1989
4,7257,1989-12-31,MERCK & CO INC,12,0.504655,6756.7,0.017435,0.251809,0.282283,0.591102,0.187310,0.187310,0.198899,0.221321,0.969482,0.035728,0.111075,0.409061,0.204345,1989


In [3]:
data.shape

(5491, 20)

In [4]:
data["Accruals"] = data["Net Income"] - data["Net Cash Flow from Operations"]
data = data[data["Accruals"] < 1]
data.shape


(5439, 21)

In [5]:
data = data.sort_values(["gvkey", "Year"])
data["Revenue Change"] = data.groupby("gvkey")["Total Revenue"].diff()
data["Prior Revenue Change"] = data.groupby("gvkey")["Revenue Change"].shift(1)
data["Prior Revenue"] = data.groupby("gvkey")["Total Revenue"].shift(1)
data = data.sort_values(["gvkey", "Total Trade Receivables"])
data["Trade Receivables Change"] = data.groupby("gvkey")["Total Trade Receivables"].diff()
data.shape

(5439, 25)

In [6]:
data["Cash Revenue Growth"] = data["Revenue Change"] - data["Trade Receivables Change"]

In [7]:
data = data[data["Cash Revenue Growth"].notna()]
data = data[data["Gross Property Plant and Equipment"].notna()]
data.head()

,gvkey,Date,Company Name,Fiscal Year,Total Current Assets,Total Assets,Total Long-Term Debt,Accumulated Depreciation,Total Current Liabilities,Gross Property Plant and Equipment,...,R&D Expense,SG&A Expense,Net Cash Flow from Operations,Year,Accruals,Revenue Change,Prior Revenue Change,Prior Revenue,Trade Receivables Change,Cash Revenue Growth
3805,6266,2015-12-31,JOHNSON & JOHNSON,12,0.451312,133411.0,0.096371,0.155482,0.207981,0.274700,...,0.069485,0.226735,0.144508,2015,-0.029008,-0.041648,0.029436,0.566897,0.002834,-0.044482
5458,6266,2024-12-31,JOHNSON & JOHNSON,12,0.310337,180104.0,0.170185,0.156854,0.279400,0.276884,...,0.096850,0.221466,0.134733,2024,-0.056634,-0.015071,0.001544,0.508236,0.001950,-0.017021
3995,6266,2016-12-31,JOHNSON & JOHNSON,12,0.460540,141208.0,0.158929,0.154814,0.186158,0.267499,...,0.064614,0.205654,0.132903,2016,-0.015771,-0.016142,-0.041648,0.525249,0.000441,-0.016583
3763,6266,2014-12-31,JOHNSON & JOHNSON,12,0.452345,131119.0,0.115330,0.156796,0.191315,0.279784,...,0.066138,0.230539,0.140872,2014,-0.016382,0.029436,-0.016520,0.537461,0.000929,0.028506
5045,6266,2021-12-31,JOHNSON & JOHNSON,12,0.335016,182018.0,0.164736,0.157770,0.248470,0.266891,...,0.085783,0.215781,0.128614,2021,-0.013911,0.043002,-0.048062,0.472195,0.000185,0.042816


In [8]:
frame = data[ data["Year"] == 2019 ]
frame.shape

(163, 26)

In [9]:
import statsmodels.formula.api as smf

regression_formula = 'Q("Accruals") ~ Q("Cash Revenue Growth") + Q("Gross Property Plant and Equipment")'
model = smf.ols(formula=regression_formula, data= frame).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:          Q("Accruals")   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                 -0.006
Method:                 Least Squares   F-statistic:                    0.5304
Date:                Mon, 02 Feb 2026   Prob (F-statistic):              0.589
Time:                        15:23:40   Log-Likelihood:                -404.01
No. Observations:                 163   AIC:                             814.0
Df Residuals:                     160   BIC:                             823.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

In [10]:
#XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

In [11]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1) Load the scaled file – the one you just showed
df = pd.read_csv("Accruals Workable.csv")

# 2) Recompute Accruals from the already scaled Net Income and CFO
df["Accruals"] = df["Net Income"] - df["Net Cash Flow from Operations"]

# 3) Cash Revenue Growth and PPE (already scaled by assets in your CSV)
df = df.sort_values(["gvkey", "Year"])
df["Cash_Rev_Growth"] = df.groupby("gvkey")["Total Revenue"].diff()
df["PPE_scaled"] = df["Gross Property Plant and Equipment"]   # already / Assets in your file

# 4) Keep only the columns we need, drop rows with missing values
sub = df[["Accruals", "Cash_Rev_Growth", "PPE_scaled"]].dropna()


In [12]:
print(sub["Accruals"].describe())
print(sub["Cash_Rev_Growth"].describe())
print(sub["PPE_scaled"].describe())


count    4953.000000
mean       -2.404174
std       103.907838
min     -7209.500000
25%        -0.217356
50%        -0.084657
75%        -0.019734
max       458.000000
Name: Accruals, dtype: float64
count    4953.000000
mean       -0.004736
std         4.862971
min      -240.740741
25%        -0.029982
50%         0.000000
75%         0.060794
max       240.740741
Name: Cash_Rev_Growth, dtype: float64
count    4953.000000
mean        0.252974
std         0.511538
min         0.000000
25%         0.038536
50%         0.134163
75%         0.317486
max        19.324324
Name: PPE_scaled, dtype: float64


In [13]:
# X matrix and y vector
X = sub[["Cash_Rev_Growth", "PPE_scaled"]]
X = sm.add_constant(X)     # adds Intercept
y = sub["Accruals"]

model = sm.OLS(y, X).fit()
print(model.summary())



                            OLS Regression Results                            
Dep. Variable:               Accruals   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.008
Method:                 Least Squares   F-statistic:                     20.73
Date:                Mon, 02 Feb 2026   Prob (F-statistic):           1.09e-09
Time:                        15:23:42   Log-Likelihood:                -30006.
No. Observations:                4953   AIC:                         6.002e+04
Df Residuals:                    4950   BIC:                         6.004e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -2.2010      1.641     

In [14]:
#TESTTESTTESTTESTTEST

In [15]:
info = pd.read_csv("Accounting.csv")
info.head()

,gvkey,Date,Company Name,Fiscal Year,Total Current Assets,Total Assets,Total Long-Term Debt,Accumulated Depreciation,Total Current Liabilities,Gross Property Plant and Equipment,Total Receivables,Total Trade Receivables,COGS,Net Income,Total Revenue,Advertising Expense,R&D Expense,SG&A Expense,Net Cash Flow from Operations
0,6266,12/31/1980,JOHNSON & JOHNSON,12,1971.000,3342.501,70.1,614.6,773.5,1776.5,643.8,643.8,2288.400,400.7,4837.379,233.5,232.8,1794.2,NaN
1,6266,12/31/1981,JOHNSON & JOHNSON,12,2202.200,3820.400,91.7,678.0,881.2,2013.6,705.9,705.9,2498.901,467.6,5399.000,270.6,282.9,2030.6,NaN
2,6266,12/31/1982,JOHNSON & JOHNSON,12,2253.101,4209.570,142.2,733.5,900.2,2311.4,758.1,758.1,2274.701,473.4,5760.871,307.7,363.2,2612.0,NaN
3,6266,12/31/1983,JOHNSON & JOHNSON,12,2457.101,4461.496,195.6,801.5,923.8,2469.7,836.4,836.4,2262.000,489.0,5972.871,301.7,405.1,2758.0,NaN
4,6266,12/31/1984,JOHNSON & JOHNSON,12,2513.699,4541.371,224.8,898.7,1042.4,2619.3,877.3,877.3,2243.099,514.5,6124.496,336.6,421.2,2909.6,NaN


In [16]:
import pandas as pd

info["Date"] = pd.to_datetime(info["Date"])
info["Year"] = info["Date"].dt.year
info.head()


,gvkey,Date,Company Name,Fiscal Year,Total Current Assets,Total Assets,Total Long-Term Debt,Accumulated Depreciation,Total Current Liabilities,Gross Property Plant and Equipment,Total Receivables,Total Trade Receivables,COGS,Net Income,Total Revenue,Advertising Expense,R&D Expense,SG&A Expense,Net Cash Flow from Operations,Year
0,6266,1980-12-31,JOHNSON & JOHNSON,12,1971.000,3342.501,70.1,614.6,773.5,1776.5,643.8,643.8,2288.400,400.7,4837.379,233.5,232.8,1794.2,NaN,1980
1,6266,1981-12-31,JOHNSON & JOHNSON,12,2202.200,3820.400,91.7,678.0,881.2,2013.6,705.9,705.9,2498.901,467.6,5399.000,270.6,282.9,2030.6,NaN,1981
2,6266,1982-12-31,JOHNSON & JOHNSON,12,2253.101,4209.570,142.2,733.5,900.2,2311.4,758.1,758.1,2274.701,473.4,5760.871,307.7,363.2,2612.0,NaN,1982
3,6266,1983-12-31,JOHNSON & JOHNSON,12,2457.101,4461.496,195.6,801.5,923.8,2469.7,836.4,836.4,2262.000,489.0,5972.871,301.7,405.1,2758.0,NaN,1983
4,6266,1984-12-31,JOHNSON & JOHNSON,12,2513.699,4541.371,224.8,898.7,1042.4,2619.3,877.3,877.3,2243.099,514.5,6124.496,336.6,421.2,2909.6,NaN,1984


In [17]:
crunch = info[info["Year"] == 2019 ]
crunch.head()
crunch.shape

(219, 20)

In [18]:
crunch.isna().sum()

gvkey                                   0
Date                                    0
Company Name                            0
Fiscal Year                             0
Total Current Assets                    6
Total Assets                            6
Total Long-Term Debt                    7
Accumulated Depreciation               21
Total Current Liabilities               6
Gross Property Plant and Equipment     22
Total Receivables                       8
Total Trade Receivables                14
COGS                                    6
Net Income                              6
Total Revenue                           6
Advertising Expense                   181
R&D Expense                            27
SG&A Expense                           71
Net Cash Flow from Operations           6
Year                                    0
dtype: int64

In [19]:
cols = [
    'Total Revenue',
    'Gross Property Plant and Equipment',
    'R&D Expense',
    'Total Trade Receivables',
    'Net Cash Flow from Operations'
]

# DROP rows where any column in cols is NA
crunch = crunch.dropna(subset=cols)


In [20]:
crunch.shape

(172, 20)

In [21]:
crunch.isna().sum()

gvkey                                   0
Date                                    0
Company Name                            0
Fiscal Year                             0
Total Current Assets                    0
Total Assets                            0
Total Long-Term Debt                    1
Accumulated Depreciation                0
Total Current Liabilities               0
Gross Property Plant and Equipment      0
Total Receivables                       2
Total Trade Receivables                 0
COGS                                    0
Net Income                              0
Total Revenue                           0
Advertising Expense                   140
R&D Expense                             0
SG&A Expense                           55
Net Cash Flow from Operations           0
Year                                    0
dtype: int64

In [22]:
crunch = crunch[crunch["Total Assets"] != 0 ]

In [23]:
#Properly scaled Jones Model

In [24]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# -------------------------------
# 1. Load data
# -------------------------------
df = pd.read_csv("Accruals Workable.csv")

# Ensure proper sorting for time operations
df = df.sort_values(["gvkey", "Year"])

# -------------------------------
# 2. Create lagged total assets
# -------------------------------
# Replace "Total Assets" with your exact Compustat column name if different (often 'at')
df["Lag_Assets"] = df.groupby("gvkey")["Total Assets"].shift(1)

# Drop rows where lagged assets are missing or zero
df = df[(df["Lag_Assets"].notna()) & (df["Lag_Assets"] != 0)]


# Compute total accruals (scaled)

df["Accruals_scaled"] = (
    df["Net Income"] - df["Net Cash Flow from Operations"]
) / df["Lag_Assets"]


# Compute Modified Jones regressors (scaled)


df["Delta_Revenue"] = df.groupby("gvkey")["Total Revenue"].diff()
df["Delta_AR"] = df.groupby("gvkey")["Total Trade Receivables"].diff()

df["Cash_Rev_Growth"] = (
    df["Delta_Revenue"] - df["Delta_AR"]
) / df["Lag_Assets"]

df["PPE_scaled"] = (
    df["Gross Property Plant and Equipment"]
) / df["Lag_Assets"]

# Keep usable observations

reg_df = df[
    ["Accruals_scaled", "Cash_Rev_Growth", "PPE_scaled"]
].dropna()

#  Winsorize (recommended in accounting research)

def winsorize(s, p=0.01):
    return s.clip(lower=s.quantile(p), upper=s.quantile(1 - p))

for col in ["Accruals_scaled", "Cash_Rev_Growth", "PPE_scaled"]:
    reg_df[col] = winsorize(reg_df[col], 0.01)


#  Run Modified Jones regression

model = smf.ols(
    "Accruals_scaled ~ Cash_Rev_Growth + PPE_scaled",
    data=reg_df
).fit(cov_type="HC3")

print(model.summary())


                            OLS Regression Results                            
Dep. Variable:        Accruals_scaled   R-squared:                       0.336
Model:                            OLS   Adj. R-squared:                  0.335
Method:                 Least Squares   F-statistic:                     59.58
Date:                Mon, 02 Feb 2026   Prob (F-statistic):           2.94e-26
Time:                        15:23:42   Log-Likelihood:                -14312.
No. Observations:                4426   AIC:                         2.863e+04
Df Residuals:                    4423   BIC:                         2.865e+04
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -0.2340      0.079     

In [25]:
df.columns

Index(['gvkey', 'Date', 'Company Name', 'Fiscal Year', 'Total Current Assets',
       'Total Assets', 'Total Long-Term Debt', 'Accumulated Depreciation',
       'Total Current Liabilities', 'Gross Property Plant and Equipment',
       'Total Receivables', 'Total Trade Receivables', 'COGS', 'Net Income',
       'Total Revenue', 'Advertising Expense', 'R&D Expense', 'SG&A Expense',
       'Net Cash Flow from Operations', 'Year', 'Lag_Assets',
       'Accruals_scaled', 'Delta_Revenue', 'Delta_AR', 'Cash_Rev_Growth',
       'PPE_scaled'],
      dtype='object')

In [26]:
#ROA added:

In [27]:
# ROA (performance measure)
df["ROA"] = df["Net Income"] / df["Lag_Assets"]

# Keep needed variables
reg_df = df[
    ["Accruals_scaled", "Cash_Rev_Growth", "PPE_scaled", "ROA"]
].dropna()

# Winsorize (recommended)
def winsorize(s, p=0.01):
    return s.clip(s.quantile(p), s.quantile(1 - p))

for col in reg_df.columns:
    reg_df[col] = winsorize(reg_df[col])

# Performance-matched Jones (Kothari et al.)
model_roa = smf.ols(
    "Accruals_scaled ~ Cash_Rev_Growth + PPE_scaled + ROA",
    data=reg_df
).fit(cov_type="HC3")

print(model_roa.summary())


                            OLS Regression Results                            
Dep. Variable:        Accruals_scaled   R-squared:                       0.942
Model:                            OLS   Adj. R-squared:                  0.942
Method:                 Least Squares   F-statistic:                     726.2
Date:                Mon, 02 Feb 2026   Prob (F-statistic):               0.00
Time:                        15:23:42   Log-Likelihood:                -8905.6
No. Observations:                4426   AIC:                         1.782e+04
Df Residuals:                    4422   BIC:                         1.784e+04
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept           0.0166      0.020     

In [28]:
#R&D and intangible Bias test:

In [29]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


#Scale R&D by lagged assets

df["RD_scaled"] = df["R&D Expense"] / df["Lag_Assets"]

# Keep regression variables
reg_df = df[
    ["Accruals_scaled", "Cash_Rev_Growth", "PPE_scaled", "RD_scaled"]
].dropna()


# 3. Winsorize (standard)

def winsorize(s, p=0.01):
    return s.clip(s.quantile(p), s.quantile(1 - p))

for col in reg_df.columns:
    reg_df[col] = winsorize(reg_df[col], 0.01)


# Run augmented Jones regression
model_rd = smf.ols(
    "Accruals_scaled ~ Cash_Rev_Growth + PPE_scaled + RD_scaled",
    data=reg_df
).fit(cov_type="HC3")

print(model_rd.summary())


                            OLS Regression Results                            
Dep. Variable:        Accruals_scaled   R-squared:                       0.563
Model:                            OLS   Adj. R-squared:                  0.562
Method:                 Least Squares   F-statistic:                     74.55
Date:                Mon, 02 Feb 2026   Prob (F-statistic):           4.94e-47
Time:                        15:23:42   Log-Likelihood:                -13386.
No. Observations:                4426   AIC:                         2.678e+04
Df Residuals:                    4422   BIC:                         2.681e+04
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept           0.0553      0.051     

In [30]:
#Working Capital vs long term accruals test:

In [31]:
import pandas as pd
import statsmodels.formula.api as smf

# Ensure correct sorting
df = df.sort_values(["gvkey", "Year"])


#Compute changes

df["Delta_CA"] = df.groupby("gvkey")["Total Current Assets"].diff()
df["Delta_CL"] = df.groupby("gvkey")["Total Current Liabilities"].diff()
df["Delta_AR"] = df.groupby("gvkey")["Total Trade Receivables"].diff()


# Working capital accruals (scaled)

df["WCA_scaled"] = (
    df["Delta_CA"] - df["Delta_CL"] - df["Delta_AR"]
) / df["Lag_Assets"]


# Long-term accruals proxy (scaled)

df["LTA_scaled"] = (
    - df["Accumulated Depreciation"]
) / df["Lag_Assets"]

#Keep usable observations

reg_df = df[
    [
        "WCA_scaled",
        "LTA_scaled",
        "Cash_Rev_Growth",
        "PPE_scaled"
    ]
].dropna()

# Winsorize

def winsorize(s, p=0.01):
    return s.clip(s.quantile(p), s.quantile(1 - p))

for col in reg_df.columns:
    reg_df[col] = winsorize(reg_df[col], 0.01)

#Regression: Working capital accruals

model_wca = smf.ols(
    "WCA_scaled ~ Cash_Rev_Growth + PPE_scaled",
    data=reg_df
).fit(cov_type="HC3")

print("=== Working Capital Accruals Model ===")
print(model_wca.summary())

# Regression: Long-term accruals

model_lta = smf.ols(
    "LTA_scaled ~ Cash_Rev_Growth + PPE_scaled",
    data=reg_df
).fit(cov_type="HC3")

print("\n=== Long-Term Accruals Model ===")
print(model_lta.summary())


=== Working Capital Accruals Model ===
                            OLS Regression Results                            
Dep. Variable:             WCA_scaled   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     3.623
Date:                Mon, 02 Feb 2026   Prob (F-statistic):             0.0268
Time:                        15:23:42   Log-Likelihood:                -13631.
No. Observations:                4420   AIC:                         2.727e+04
Df Residuals:                    4417   BIC:                         2.729e+04
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Int

In [32]:
#SG&A

In [33]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Load + sort
df = pd.read_csv("Accruals Workable.csv")
df = df.sort_values(["gvkey", "Year"])

# Build lagged assets 
df["Lag_Assets"] = df.groupby("gvkey")["Total Assets"].shift(1)
df = df[(df["Lag_Assets"].notna()) & (df["Lag_Assets"] != 0)]

# Build scaled accruals (TA/A_{t-1}) 
df["Accruals_scaled"] = (df["Net Income"] - df["Net Cash Flow from Operations"]) / df["Lag_Assets"]

# Build Modified Jones regressors (scaled) 
df["Delta_Revenue"] = df.groupby("gvkey")["Total Revenue"].diff()
df["Delta_AR"] = df.groupby("gvkey")["Total Trade Receivables"].diff()

df["Cash_Rev_Growth"] = (df["Delta_Revenue"] - df["Delta_AR"]) / df["Lag_Assets"]
df["PPE_scaled"] = df["Gross Property Plant and Equipment"] / df["Lag_Assets"]

# SG&A intensity (scaled)
df["SGA_scaled"] = df["SG&A Expense"] / df["Lag_Assets"]

# Winsorization helper
def winsorize(s, p=0.01):
    lo, hi = s.quantile(p), s.quantile(1 - p)
    return s.clip(lo, hi)

base_df = df[["Accruals_scaled", "Cash_Rev_Growth", "PPE_scaled"]].dropna().copy()
for c in base_df.columns:
    base_df[c] = winsorize(base_df[c], 0.01)

jones_model = smf.ols(
    "Accruals_scaled ~ Cash_Rev_Growth + PPE_scaled",
    data=base_df
).fit(cov_type="HC3")

print("\n=== Baseline Modified Jones ===")
print(jones_model.summary())

# Save residuals back (align by index)
df.loc[base_df.index, "Jones_residual"] = jones_model.resid


# SG&A-augmented Jones (main SG&A test)

aug_df = df[["Accruals_scaled", "Cash_Rev_Growth", "PPE_scaled", "SGA_scaled"]].dropna().copy()
for c in aug_df.columns:
    aug_df[c] = winsorize(aug_df[c], 0.01)

model_sga = smf.ols(
    "Accruals_scaled ~ Cash_Rev_Growth + PPE_scaled + SGA_scaled",
    data=aug_df
).fit(cov_type="HC3")

print("\n=== Jones + SG&A Intensity ===")
print(model_sga.summary())


# Does SG&A explain what Jones misses?

resid_df = df[["Jones_residual", "SGA_scaled"]].dropna().copy()
for c in resid_df.columns:
    resid_df[c] = winsorize(resid_df[c], 0.01)

sga_resid_test = smf.ols(
    "Jones_residual ~ SGA_scaled",
    data=resid_df
).fit(cov_type="HC3")

print("\n=== SG&A Explains Jones Residuals? ===")
print(sga_resid_test.summary())



=== Baseline Modified Jones ===
                            OLS Regression Results                            
Dep. Variable:        Accruals_scaled   R-squared:                       0.336
Model:                            OLS   Adj. R-squared:                  0.335
Method:                 Least Squares   F-statistic:                     59.58
Date:                Mon, 02 Feb 2026   Prob (F-statistic):           2.94e-26
Time:                        15:23:42   Log-Likelihood:                -14312.
No. Observations:                4426   AIC:                         2.863e+04
Df Residuals:                    4423   BIC:                         2.865e+04
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept

In [34]:
#Receivables Manipulation check:

In [35]:
import statsmodels.formula.api as smf
import pandas as pd

# Baseline Jones model
jones_model = smf.ols(
    "Accruals_scaled ~ Cash_Rev_Growth + PPE_scaled",
    data=df
).fit(cov_type="HC3")

# Save residuals
df["Jones_residual"] = jones_model.resid
# Scale Delta_AR by lagged assets
df["Delta_AR_scaled"] = df["Delta_AR"] / df["Lag_Assets"]

# Keep usable observations
test_df = df[
    ["Jones_residual", "Delta_AR_scaled"]
].dropna()
def winsorize(s, p=0.01):
    return s.clip(s.quantile(p), s.quantile(1 - p))

test_df["Jones_residual"] = winsorize(test_df["Jones_residual"])
test_df["Delta_AR_scaled"] = winsorize(test_df["Delta_AR_scaled"])
corr = test_df["Jones_residual"].corr(test_df["Delta_AR_scaled"])
print("Correlation:", corr)
ar_test = smf.ols(
    "Jones_residual ~ Delta_AR_scaled",
    data=test_df
).fit(cov_type="HC3")

print(ar_test.summary())


Correlation: -0.0165138051677558
                            OLS Regression Results                            
Dep. Variable:         Jones_residual   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                    0.1606
Date:                Mon, 02 Feb 2026   Prob (F-statistic):              0.689
Time:                        15:23:42   Log-Likelihood:                -18101.
No. Observations:                4426   AIC:                         3.621e+04
Df Residuals:                    4424   BIC:                         3.622e+04
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept

In [36]:
df.shape

(4953, 29)